In [0]:
%run ../transform/operaciones_replica

In [0]:
# Coordenadas lógicas de origen (Trusted Layer) y destino (Analytical Layer)
TABLA_ORIGEN = "platform_dev.silver_byma.operaciones_replica"
tabla_destino = "platform_dev.gold_byma.dim_cliente"

# Configuración homogenizada de parámetros de entrada para la orquestación centralizada
dbutils.widgets.text("full_load", "0")
full_load = int(dbutils.widgets.get("full_load"))
carga_full = (full_load == 1)

import logging
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("dimension_clientes")

In [0]:
try:
    # Programación Defensiva: Si la tabla analítica no existe por un despliegue en frío, 
    # el script levanta el DDL general y fuerza la carga full para poblar el maestro.
    if not spark.catalog.tableExists(tabla_destino):
        logger.warning(
            f"La tabla {tabla_destino} no existe. "
            "Se ejecutará el notebook de creación de tablas."
        )
        dbutils.notebook.run("../DDL/creacion_tablas", 0)
        carga_full = True
        logger.info("Tablas y esquemas creados correctamente. Se realizará una carga full.")

    elif carga_full:
        logger.warning(f"Carga full forzada manualmente para {tabla_destino}.")

    else:
        logger.info(f"La tabla {tabla_destino} existe.")
        
        # Resiliencia: Si la tabla existe físicamente pero sufrió un truncado externo,
        # cambiamos dinámicamente a modo histórico para evitar inconsistencias lógicas.
        tiene_datos = spark.sql(f"SELECT 1 FROM {tabla_destino} LIMIT 1").take(1) != []

        if tiene_datos:
            logger.info(f"La tabla {tabla_destino} contiene datos. Carga incremental activa.")
        else:
            carga_full = True
            logger.warning(
                f"La tabla {tabla_destino} está vacía. Se realizará una carga full."
            )

except Exception:
    logger.exception(f"Error al validar o crear la tabla {tabla_destino}")
    raise

In [0]:
logger.info(f"Iniciando lectura de la fuente: {TABLA_ORIGEN}")
df_origen = spark.table(TABLA_ORIGEN)

if not carga_full:
    # Recuperación segura del Watermark temporal gestionando excepciones de Cold Start
    fila_control = spark.sql(f"""
        SELECT ultima_fecha_procesada
        FROM platform_dev.bronce_byma.control_cargas
        WHERE proceso = '{tabla_destino}'
    """).first()

    if fila_control is not None and fila_control["ultima_fecha_procesada"] is not None:
        ultima_fecha = fila_control["ultima_fecha_procesada"]
        logger.info(f"Carga incremental para {tabla_destino}. Última partición procesada: {ultima_fecha}")
    else:
        ultima_fecha = "1970-01-01"
        logger.warning(f"No se encontró registro de control (Arranque en frío). Base: {ultima_fecha}")

    # OPTIMIZACIÓN CLAVE DE INGENIERÍA: Para evitar recalcular el histórico entero de la empresa,
    # 1. Identificamos qué clientes operaron únicamente en el rango de los archivos nuevos (Deltas).
    df_clientes_activos = df_origen.filter(
        F.col("fecha_particion") > F.lit(ultima_fecha)
    ).select("id_cliente").distinct()

    # 2. Traemos toda la historia transaccional pero EXCLUSIVAMENTE para ese subconjunto de clientes activos.
    # Esto reduce drásticamente el volumen de datos en memoria y optimiza el shuffle en el clúster.
    df_base_calculo = df_origen.join(df_clientes_activos, "id_cliente", "inner")
    logger.info("Aplicada estrategia de optimización por delta de clientes activos.")

else:
    logger.info(f"Carga full para {tabla_destino}. No se aplican filtros de partición.")
    df_base_calculo = df_origen

# Transformaciones Analíticas y Aplicación de Reglas de Negocio
df_transformado = (
    df_base_calculo
    .groupBy("id_cliente")
    .agg(F.count("id_transaccion").alias("total_operaciones"))
    .filter("id_cliente IS NOT NULL")
    # Regla de Negocio Core: Clasificación dinámica basada en la densidad transaccional
    .withColumn(
        "segmento_actividad",
        F.when(F.col("total_operaciones") > 30, "Alta frecuencia")
         .when(F.col("total_operaciones") >= 5, "Inversor")
         .otherwise("Esporadico")
    )
    .select("id_cliente", "segmento_actividad")
)

# Exposición de datos transformados en el catálogo temporal de memoria para Spark SQL
df_transformado.createOrReplaceTempView("v_clientes_preparados")
logger.info("Transformaciones completadas y vista temporal creada.")

In [0]:
cantidad_registros = df_transformado.count()

try:
    if carga_full:
        logger.warning(f"Carga full: Iniciando sobreescritura de {cantidad_registros} registros en {tabla_destino}")

        # PROTECCIÓN IDENTITY: Se declaran explícitamente las columnas funcionales de destino ommitiendo 'cliente_sk'.
        # Esto previene que el motor Delta destruya la secuencia autogenerada, asignando IDs estables desde 1.
        spark.sql(f"""
            INSERT OVERWRITE {tabla_destino} (id_cliente, segmento_actividad, _created_at)
            SELECT id_cliente, segmento_actividad, current_timestamp()
            FROM v_clientes_preparados
        """)
        logger.info(f"Carga full finalizada: {cantidad_registros} registros insertados.")

    else:
        if cantidad_registros == 0:
            logger.info("Carga incremental: No se detectaron clientes con actividad nueva.")
        else:
            logger.info(f"Carga incremental (MERGE): Procesando {cantidad_registros} clientes modificados/nuevos.")

            # UPSERT DIMENSIONAL DEFENSIVO: Si el cliente ya existe y cambió su segmento, se actualiza el atributo.
            # Al no tocar el registro si no cambió, preservamos de forma inmutable la 'cliente_sk' original
            # para que las transacciones históricas en la tabla de hechos no apunten a claves huérfanas.
            spark.sql(f"""
                MERGE INTO {tabla_destino} AS target
                USING v_clientes_preparados AS source
                ON target.id_cliente = source.id_cliente

                WHEN MATCHED AND target.segmento_actividad <> source.segmento_actividad THEN
                  UPDATE SET 
                    target.segmento_actividad = source.segmento_actividad,
                    target._created_at = current_timestamp()

                WHEN NOT MATCHED THEN
                  INSERT (id_cliente, segmento_actividad, _created_at)
                  VALUES (source.id_cliente, source.segmento_actividad, current_timestamp())
            """)
            logger.info(f"Carga incremental finalizada con éxito para {cantidad_registros} registros.")

except Exception:
    logger.exception(f"Error durante la escritura en {tabla_destino}")
    raise

In [0]:
# Extraemos el límite superior del lote de origen para registrar el progreso del pipeline
ultima_fecha_origen = df_origen.agg(F.max("fecha_particion").alias("max_fecha")).first()["max_fecha"]

if ultima_fecha_origen is not None and cantidad_registros > 0:
    try:
        logger.info(f"Actualizando watermark del proceso {tabla_destino} con fecha de corte: {ultima_fecha_origen}")

        # Impacto transaccional en la matriz maestro de control de cargas
        spark.sql(f"""
        MERGE INTO platform_dev.bronce_byma.control_cargas AS target
        USING (
            SELECT
                '{tabla_destino}' AS proceso,
                DATE('{ultima_fecha_origen}') AS ultima_fecha_procesada,
                current_timestamp() AS fecha_actualizacion
        ) AS source
        ON target.proceso = source.proceso

        WHEN MATCHED THEN UPDATE SET
            target.ultima_fecha_procesada = source.ultima_fecha_procesada,
            target.fecha_actualizacion = source.fecha_actualizacion

        WHEN NOT MATCHED THEN INSERT (proceso, ultima_fecha_procesada, fecha_actualizacion)
        VALUES (source.proceso, source.ultima_fecha_procesada, source.fecha_actualizacion)
        """)
        logger.info(f"Watermark actualizado correctamente para {tabla_destino}.")

    except Exception:
        logger.exception(f"Error al actualizar la tabla de control para {tabla_destino}")
        raise
else:
    logger.warning("No se modificó el watermark de control debido a que no hubo procesamiento incremental.")